## Goals
Answer the following questions:
1. Demand Impact: How does congestion pricing affect trip volume in/out of congestion zones?
2. Price Sensitivity: What is the elasticity of demand to fare changes?
3. Behavioral Changes: Do travelers adjust timing, routes, company election (Uber vs. Traditional taxi) or pickup/dropoff locations to avoid fees?
4. Revenue Optimization: What pricing strategies maintain revenue while managing demand?


## Step 1 - Data/Feature Engineering Process (Summary)

### Step 1.1 Download data:

<div style="border: 1px solid #e74c3c; padding: 12px 16px; border-radius: 4px; background-color: #fdf0f0;">
<span style="color: #e74c3c; font-weight: bold;"> NOTE</span>

Due to the extensive nature of the data engineering process, this notebook presents only the **pseudo-code and key results** of each step. The full production code — including downloads, transformations, cleaning, and merging pipelines — is documented separately.

To inspect the detailed implementation, please refer to **`annex_a_data_engineering.ipynb`**.
</div>

### Step 1.1.1 Weather data
#### Step 1.1.1.1 Download weather data:

We start by downloading unstructured data from https://open-meteo.com/en/docs/historical-weather-api

### Step 1.1.1.2 transform and encode weather data using WMO codes

### 1.1.1.3 Basic stats and Nan verification for weather

### Step 1.1.2 Download and initial transform for Uber and Yellow Data

#### 1.1.2.1 UBER data download

Uber data comes in a dataset called High Volume FHV Trip Records; let's download from 2024/01/01 to 2025/11/30

##### 1.1.2.2 Filtering UBER

Uber comes with other provider as LYFT, so it is neccesary to filter only uber category: hvfhs_license_num  HV0003: Uber.

#### 1.1.2.3 Transform UBER

Transform filtered Uber data to match Yellow taxi format.

Transformations:
- Create provider = 2 (Uber identifier)
- Rename: pickup_datetime -> pickup_datetime 
- Rename: dropoff_datetime -> dropoff_datetime 
- Rename: trip_miles -> trip_distance
- Rename: trip_time (minutes) -> trip_time
- Rename: base_passenger_fare -> fare_amount
- Rename: sales_tax -> tax
- Rename: tolls -> tolls_amount
- Create: total_amount = base_passenger_fare + tolls + bcf + sales_tax + congestion_surcharge + airport_fee + tips
- Keep: congestion_surcharge, airport_fee
- Handle: cbd_congestion_fee (not in Uber data - set to 0)
- Delete: All other Uber-specific columns

Final column order (must match Yellow):
1. pickup_datetime, 2. provider, 3. dropoff_datetime,
4. PULocationID, 5. DOLocationID, 6. trip_distance, 7. trip_time,
8. fare_amount, 9. tax, 10. tolls_amount, 11. total_amount,
12. congestion_surcharge, 13. airport_fee, 14. cbd_congestion_fee


##### 1.1.2.4 Yellow taxi data download

##### 1.1.2.5 Transform Yellow taxi

Transform NYC Yellow Taxi raw data into a standardized format compatible with Uber data.

Transformations applied to each file:
1. provider        : Created with value 1 (Yellow Taxi identifier)
2. trip_time       : Calculated in seconds (tpep_dropoff_datetime - tpep_pickup_datetime)
3. tax             : Renamed from mta_tax
4. airport_fee     : Renamed from Airport_fee (capital A in source)
5. cbd_congestion_fee : Set to 0.0 for 2024 files (CBD pricing started January 2025)

Columns removed: VendorID, passenger_count, RatecodeID, store_and_fwd_flag,
                 payment_type, extra, mta_tax, tip_amount, improvement_surcharge, Airport_fee

Final column order (14 columns, must match Uber format):
  tpep_pickup_datetime, provider, tpep_dropoff_datetime, PULocationID, DOLocationID,
  trip_distance, trip_time, fare_amount, tax, tolls_amount, total_amount,
  congestion_surcharge, airport_fee, cbd_congestion_fee

### 1.2 Data Engineering

#### 1.2.1 Uber data engineering

Clean and enhance Uber transformed data.

Operations:
1. Fix trip_time bug (recalculate from timestamps, original was 60x too large)
2. Remove negative values (trip_time, trip_distance, fare_amount, total_amount)
3. Remove extreme outliers
4. Remove invalid speeds
5. Remove invalid LocationIDs
6. Add calculated fields: in_cbd_zone, speed_mph, cost_per_mile
7. Calculate cbd_congestion_fee ONLY for trips on/after January 5, 2025
8. Keep datetime columns in standard format

#### 1.2.2 Yellow Taxi Data Engineering

Cleaning operations (applied in order):
1. Recalculate trip_time (seconds) from tpep_pickup/dropoff_datetime timestamps
2. Remove rows with negative values in: trip_time, trip_distance, fare_amount, total_amount
3. Remove extreme outliers:
      trip_distance > 100 miles | trip_time > 3 hours (10800s)
      fare_amount > $500        | total_amount > $600
4. Calculate speed_mph (vectorized); remove trips outside 3–80 mph range
5. Remove rows with invalid LocationIDs (valid range: 1–263)
6. Add in_cbd_zone: 1 if PULocationID is in Manhattan CBD zone, else 0
7. Calculate cost_per_mile = total_amount / trip_distance (vectorized)
8. Calculate cbd_congestion_fee (vectorized):
      Before Jan 5, 2025 → $0.00
      On/after Jan 5, 2025, CBD pickup  → $1.25
      On/after Jan 5, 2025, non-CBD     → $0.75
9. Rename tpep_pickup_datetime → pickup_datetime
         tpep_dropoff_datetime → dropoff_datetime

#### 1.2.3 Merge weather data to Uber

1. Auto-detect years from Uber filenames
2. Find corresponding weather CSV for each year
3. Round pickup_datetime down to nearest hour
4. Left join Uber rows with weather data on hourly timestamp
5. Weather columns added: temperature, precipitation, windspeed,
   weathercode, visibility, weather_type

#### 1.2.4 Merge weather data to Yellow taxis

1. Auto-detect years from Uber filenames
2. Find corresponding weather CSV for each year
3. Round pickup_datetime down to nearest hour
4. Left join Uber rows with weather data on hourly timestamp
5. Weather columns added: temperature, precipitation, windspeed,
   weathercode, visibility, weather_type

### 1.4 Generating monthly datasets (uber + yellow taxis)

We join in monthly datasets the data from Uber and yellow taxis

Merge Yellow Taxi and Uber monthly data files into a single combined dataset.

1. Auto-detect years from filenames (2024, 2025, or both)
2. Pair Yellow and Uber files by year-month
3. Verify schema compatibility (columns, order, provider values)
4. Concatenate Yellow (provider=1) + Uber (provider=2) per month
5. Verify row count integrity after concatenation

### 1.4.1 Data engineering/transformation pipeline

The pipeline used to produce the dataset for this project is summarized in below graphic

<div style="text-align: center;">
    <img src="data_transformation_engineering_pipeline.png" width="1200"/>
</div>

#### 1.4.2 Monthly data statistics results 

<div style="text-align: center;">
    <img src="monthly_statistics_dashboard.png" width="1200"/>
</div>

### 1.5 Final dataset

Due to the massive size of the dataset, merging all the data in just one dataset will make it extreamly difficult to process, so we perform an stratified sampling mont by month keeping the proportion in order to generate the final file

### 1.6 Segmentation & Feature Insights

We will perform some analysis to preliminary identify patterns and clusters to the final dataset;
So, in first instance we perform a K-mean algorithm

#### 1.6.1 K-means algorithm

#### Elbow method


<div style="text-align: center;">
    <img src="elbow_plot.png" width="1200"/>
</div>

#### K-mean cluster

Based in the elbow method, 6 clusters were recommended, so following plots were generated 

<div style="text-align: center;">
    <img src="pickup_cluster.png" width="1200"/>
</div>

<div style="text-align: center;">
    <img src="kmeans.png" width="1200"/>
</div>

Initial inspection indicates a significant correlation between location and ride frequency. Nevertheless, the presence of high-density clusters with irregular, non-convex shapes poses a challenge for standard clustering techniques. To identify non-obvious relationships, we must utilize algorithms capable of detecting arbitrary morphologies and varying densities, moving beyond simple distance-based partitioning.

We choose density based algorithms to try to find non linear relationships in the dataset

#### 1.6.2 DBSCAN

Our first approach to the density based cluster analysis was by using DBSCAN; however as is shown in below graphic, despite several hyperparameters adjust, the output was not clear about hidden relationships in the dataset

<div style="text-align: center;">
    <img src="dbscan_clusters.png" width="1200"/>
</div>

### 1.6.3 UMAP

Based on previous results, we change the aproximation to the cluster analysis by using an algorithm that preserves distance and relationships between the datapoints.

The results are in below graphics

<div style="text-align: center;">
    <img src="umap_clusters.png" width="1200"/>
</div>

<div style="text-align: center;">
    <img src="umap_context.png" width="1200"/>
</div>

### UMAP Analysis — Interpretation

**What is UMAP?**
UMAP (Uniform Manifold Approximation and Projection) compresses 13 numerical features into a 2D space while preserving the neighborhood structure of the data. Each point represents a single trip. **Points that appear close together share similar characteristics across all 13 features simultaneously.** Distinct "islands" in the plot indicate groups of trips with genuinely different behavioral profiles.

---

#### Plot 1 — UMAP colored by DBSCAN cluster

DBSCAN identified **7 natural trip clusters** in the UMAP embedding:

| Cluster | Size | Likely profile |
|---------|------|----------------|
| **0** (red) | ~2,273 | Typical short urban trips — low fare, outside CBD, fast turnaround |
| **1** (teal) | ~1,107 | Medium-distance trips with moderate fares |
| **2** (orange) | ~308 | Mixed/transitional trips — moderate distance and cost |
| **3** (purple) | ~364 | Atypical trips — possibly late-night, extreme weather, or unusual routes |
| **4** (yellow) | ~662 | Higher-fare trips — likely include airport fees or CBD surcharges |
| **5** (light green) | ~99 | Specialized trips — likely airport pickups/dropoffs |
| **6** (blue) | ~40 | Micro-cluster of outliers — very long trips or unusually high tolls |

The clear spatial separation between islands confirms that these groups are **structurally distinct**, not artifacts of the algorithm.

---

#### Plot 2 — UMAP colored by total fare (continuous gradient)

The color scale goes from **dark purple (cheap) → yellow (expensive)**:

- The large left island is predominantly **dark** → typical urban trips costing ~$10–20
- The right-side islands (clusters 4, 5, 6) are **bright yellow** → fares ranging $80–120
- This confirms that **fare amount is one of the primary axes of differentiation** between trip types, closely aligned with UMAP's first dimension (UMAP-1)

---

#### Plot 3 — UMAP colored by CBD zone

- **Red** = trip originated or ended inside the Congestion Pricing Zone (Manhattan, south of 60th St.)
- **Gray** = trips entirely outside the CBD

The right-side clusters are **almost entirely red**, meaning expensive long trips are strongly associated with the CBD. The large left cluster is mixed, reflecting the diversity of non-CBD trip types. This spatial pattern suggests that **CBD zone membership is a key structural driver** of trip clustering.

---

#### Plot 4 — UMAP colored by time period (Before / After congestion pricing)

- **Teal** = 2024 trips (before congestion pricing, Jan–Dec 2024)
- **Orange** = 2025 trips (after congestion pricing, Jan 2025 onward)

The 2025 trips (orange) do **not** form an isolated cluster — they are distributed across the same islands as 2024 trips. This is a meaningful finding: **congestion pricing did not create a fundamentally new trip profile**. Instead, it appears to have shifted the frequency and composition within existing behavioral groups. Further causal analysis (DiD, elasticity modeling) is needed to quantify the magnitude of that shift.

---

#### Summary

> The UMAP structure primarily reflects a **fare/distance gradient** running left to right, with short cheap urban trips on the left and expensive long-distance or CBD-heavy trips on the right. The CBD zone and airport fees emerge as the clearest structural separators between clusters. The before/after overlay suggests that post-pricing behavior occupies the same latent space as pre-pricing behavior, pointing to a **gradual behavioral adjustment** rather than a structural break.
Markdown inse